### Structured Output

In [3]:
# we can use pydantic for the structure
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

os.environ['GROQ_API_KEY'] = os.getenv('LLM_API')
model = init_chat_model(model='groq:qwen/qwen3-32b') # reasoning model

In [4]:
!uv add pydantic

Resolved 59 packages in 6ms
Checked 58 packages in 9ms


In [5]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(description="The title of the movie")
    year:int = Field(description="This year the movie was released")
    director:str = Field(description="Name of the director")
    rating:float = Field(description="The movie's rating out of 10")

In [13]:
# without structure
model.invoke("Provide details about the movie Intersteller")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Interstellar. Let me start by recalling what I know about it. It\'s a science fiction film directed by Christopher Nolan, right? The release year was 2014. The main cast includes Matthew McConaughey, Anne Hathaway, and maybe some others. The plot revolves around space exploration, time dilation, and wormholes.\n\nWait, the story is about a group of astronauts who travel through a wormhole to find a new home for humanity. The wormhole appears near Saturn, which allows them to reach other star systems. The main character, Cooper, is a former NASA pilot turned farmer, but he joins a mission with his daughter Murph. There\'s something about the concept of time slowing down due to gravity, which is a key element in the movie.\n\nI remember there\'s a part where they visit different planets, each with unique conditions. One planet has massive tidal waves, another has a time-dilated environment where one hour equal

In [7]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x73d118ceaa50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x73d118a59bd0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'Name of the director', 'type': 'string'}, 'rating': {'description': "The movie's rating out of 10", 'ty

In [9]:
details=model_with_structure.invoke("Provide details about the movie Intersteller")

In [12]:
details

Movie(title='Interstellar', year=2014, director='Christopher Nolan', rating=8.6)

### Message Output alongside parsed structure

In [15]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str = Field(...,description="The title of the movie")
    year:int = Field(...,description="This year the movie was released")
    director:str = Field(...,description="Name of the director")
    rating:float = Field(...,description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie,include_raw=True)
response = model_with_structure.invoke("Please Provide details about movie Oppenhiemer") # it will return both raw and parsed response
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Oppenheimer." Let me see what I need to do here. The available tool is the Movie function, which requires the title, year, director, and rating. First, I need to confirm that "Oppenheimer" is a real movie. Wait, I think there\'s a movie titled "Oppenheimer" directed by Christopher Nolan, released in 2023. Let me check the details. The movie is about J. Robert Oppenheimer, the father of the atomic bomb. The director is Christopher Nolan, and I believe it has a high rating. I should get the exact year and rating. The release year is 2023, and the rating on IMDb is around 8.0. So I need to structure the function call with these parameters. Make sure all required fields are included: title, year, director, and rating. Let me put that into the JSON format as specified.\n', 'tool_calls': [{'id': '6rtfkre5w', 'function': {'arguments': '{"director":"Christopher Nolan","r

In [17]:
response['raw']

AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Oppenheimer." Let me see what I need to do here. The available tool is the Movie function, which requires the title, year, director, and rating. First, I need to confirm that "Oppenheimer" is a real movie. Wait, I think there\'s a movie titled "Oppenheimer" directed by Christopher Nolan, released in 2023. Let me check the details. The movie is about J. Robert Oppenheimer, the father of the atomic bomb. The director is Christopher Nolan, and I believe it has a high rating. I should get the exact year and rating. The release year is 2023, and the rating on IMDb is around 8.0. So I need to structure the function call with these parameters. Make sure all required fields are included: title, year, director, and rating. Let me put that into the JSON format as specified.\n', 'tool_calls': [{'id': '6rtfkre5w', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8

### Nested Structure

In [24]:
from pydantic import BaseModel, Field
class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget: float | None = Field(None,description="Budget of Movie in USD.")

In [25]:
new_model = model.with_structured_output(MovieDetails)
response=new_model.invoke("Please Provide details about 3 idiots")
print(response)

title='3 Idiots' year=2009 cast=[Actor(name='Aamir Khan', role='Farhan Qureshi'), Actor(name='R. Madhavan', role='Raju Rastogi'), Actor(name='Sharman Joshi', role='Imran Chaudhary (Chandru)'), Actor(name='Boman Irani', role='Professor Rao (Pundit)')] genres=['Comedy', 'Drama'] budget=7500000.0


### Typed Dict

Run time validation is not there in Typed Dict

In [26]:
from typing import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str,...,"The title of movie"]
    year: Annotated[int,...,"Release Year of movie"]
    director: Annotated[str,...,"The Director of movie"]
    rating: Annotated[float, ..., "Rating of movie out of 10"]

In [29]:
model.with_structured_output(MovieDict).invoke("Please provide details about Intersteller")

{'director': 'Christopher Nolan',
 'rating': 8.6,
 'title': 'Interstellar',
 'year': 2014}

In [32]:
# nested typed dict
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget: float | None = Field(None,description="Budget of Movie in USD.")
model.with_structured_output(MovieDetails).invoke("Please provide details about The Social Network")

{'budget': 50000000,
 'cast': [{'name': 'Jesse Eisenberg', 'role': 'Mark Zuckerberg'},
  {'name': 'Andrew Garfield', 'role': 'Eduardo Saverin'},
  {'name': 'Emma Stone', 'role': 'Erica Albright'},
  {'name': 'Justin Timberlake', 'role': 'Sean Parker'}],
 'genres': ['Drama', 'Biography'],
 'title': 'The Social Network',
 'year': 2010}

In [33]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

In [ ]:
model.with_structured_output(MovieDetails).profile # not available for structure model

AttributeError: 'RunnableSequence' object has no attribute 'profile'

### DataClasses

A data class is a class typically containing mainly data, although it can also contain methods. It is a convenient way to define classes that are primarily used to store data without having to write boilerplate code for initialization, representation, and comparison. Created using @dataclass decorator

In [38]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact Format of a person"""
    name:str 
    email:str
    phone:str

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo
)
result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from here where name is rahul, email is rg4005450@gmail.com, and phone is 6230822583 this is an indian number so add country code too"}]
})
result

{'messages': [HumanMessage(content='Extract contact info from here where name is rahul, email is rg4005450@gmail.com, and phone is 6230822583 this is an indian number so add country code too', additional_kwargs={}, response_metadata={}, id='d3341231-957a-4b66-a543-090d5734407f'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants me to extract contact information for someone named Rahul. The details provided are the email, phone number, and they mentioned it\'s an Indian number, so I need to add the country code.\n\nFirst, I\'ll check the function signature again. The ContactInfo function requires name, email, and phone. The phone number given is 6230822583. Since it\'s an Indian number, the country code should be +91. So I\'ll format the phone number as +91 6230822583. \n\nWait, should I include the space or just append the country code? The user wrote "add country code too," so maybe just adding +91 without any spaces. Let me confirm. The original n

In [39]:
result['structured_response']

ContactInfo(name='rahul', email='rg4005450@gmail.com', phone='+916230822583')